<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-04/AI_Modul_4.7_Praktikum_Membaca_Dataset_CSV_dan_Excel.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum AI Modul 4.7: Membaca Dataset (CSV/Excel)
Program Studi Sarjana Sains Data / Kecerdasan Buatan / Teknik Pertanian  
Fakultas Teknologi Pertanian, Institut Pertanian STIPER Yogyakarta

---

### Capaian Pembelajaran Praktikum:
1. Menguasai parameter teknis `pd.read_csv()` (enkoding, delimiter, tipe data eksplisit, comment, decimal).
2. Mengekstrak dan mengonsolidasi lembar kerja ganda (*multi-sheet*) dari berkas Excel menggunakan `pd.ExcelFile`.
3. Mengimplementasikan pembacaan bongkahan (*chunking iterator*) untuk mengolah dataset besar secara hemat RAM.
4. Melakukan benchmark efisiensi I/O antara format CSV, Excel (XLSX), dan Apache Parquet.

In [ ]:
import os
import time
import tempfile
import numpy as np
import pandas as pd
import openpyxl

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"OpenPyXL version: {openpyxl.__version__}")


In [ ]:
# Setup direktori kerja sementara untuk demonstrasi I/O
data_dir = os.path.join(tempfile.gettempdir(), "praktikum_4_7_data")
os.makedirs(data_dir, exist_ok=True)

# 1. Dataset CSV standar kelapa sawit
csv_standar_path = os.path.join(data_dir, "transaksi_timbang_standar.csv")
df_standar = pd.DataFrame({
    "No_Tiket": [f"TKT-{1000+i}" for i in range(10)],
    "Afdeling": ["OA", "OB", "OC", "OD", "OA", "OB", "OC", "OD", "OA", "OB"],
    "Berat_TBS_kg": [5400, 6200, 4800, 7100, 5900, 6400, 5100, 7300, 5600, 6800],
    "Status_Mutu": ["Prima", "Standar", "Prima", "Afkir", "Prima", "Standar", "Standar", "Prima", "Prima", "Standar"]
})
df_standar.to_csv(csv_standar_path, index=False)

# 2. Dataset CSV non-standar (pemisah titik-koma, koma desimal, enkoding latin-1, komentar di awal)
csv_nonstandar_path = os.path.join(data_dir, "laporan_laboratorium_latin1.csv")
with open(csv_nonstandar_path, "w", encoding="latin-1") as f:
    f.write("# Laporan Uji FFA Laboratorium Pabrik Kelapa Sawit\n")
    f.write("# Lokasi: Stasiun Klarifikasi PKS Bukit Mas\n")
    f.write("ID_Sampel;Tanggal_Uji;Kadar_FFA;Suhu_C;Catatan\n")
    f.write("SMP-01;2026-09-01;3,45;85,2;Kondisi normal\n")
    f.write("SMP-02;2026-09-02;4,12;88,0;Minyak asam tinggi\n")
    f.write("SMP-03;2026-09-03;2,98;84,5;Kualitas prima (ekspor)\n")
    f.write("SMP-04;2026-09-04;5,30;91,1;Indikasi buah restan\n")

# 3. Dataset Excel Multi-Sheet
xlsx_path = os.path.join(data_dir, "panen_bulanan_kebun.xlsx")
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    for afd in ["Afdeling_A", "Afdeling_B", "Afdeling_C"]:
        df_afd = pd.DataFrame({
            "Tanggal": pd.date_range("2026-09-01", periods=5, freq="D"),
            "Jumlah_Tandan": np.random.randint(150, 300, 5),
            "Tonase_Panen_Ton": np.round(np.random.uniform(3.5, 7.2, 5), 2)
        })
        df_afd.to_excel(writer, sheet_name=afd, index=False)

print(f"Semua berkas pengujian berhasil di-generate di: {data_dir}")


## 1. Membaca Berkas CSV: Parameter Kritis dan Spesifikasi Skema Data
Fungsi `pd.read_csv()` dilengkapi puluhan parameter teknis. Tiga parameter paling fundamental untuk menjamin efisiensi memori dan keakuratan data adalah:
- `sep` / `delimiter`: Simbol pembatas kolom.
- `dtype`: Penetapan tipe data kolom secara eksplisit untuk mencegah overhead konversi.
- `usecols`: Seleksi subset kolom yang dibutuhkan saja (proyeksi kolom).


In [ ]:
# Membaca dataset CSV standar dengan skema tipe data eksplisit
schema_dtypes = {
    "No_Tiket": "string",
    "Afdeling": "category",
    "Berat_TBS_kg": "int32",
    "Status_Mutu": "category"
}

df_read_standar = pd.read_csv(
    csv_standar_path,
    usecols=["No_Tiket", "Afdeling", "Berat_TBS_kg", "Status_Mutu"],
    dtype=schema_dtypes
)

print("--- Skema Data Setelah Ingesti Eksplisit ---")
print(df_read_standar.info())
print("\nPreview 3 Data Teratas:")
print(df_read_standar.head(3))


## 2. Menangani Format CSV Non-Standar (Enkoding, Delimiter, Koma Desimal)
Sistem operasional di lapangan kerap menghasilkan CSV dengan:
1. Enkoding karakter non-UTF-8 (misal `latin-1` atau `cp1252`).
2. Tanda pemisah titik-koma (`;`) alih-alih koma murni.
3. Koma sebagai desimal numerik (`3,45` alih-alih `3.45`).
4. Baris metadata/komentar di awal tabel (`#`).


In [ ]:
# Membaca CSV non-standar laboratorium PKS
df_lab = pd.read_csv(
    csv_nonstandar_path,
    sep=";",
    encoding="latin-1",
    decimal=",",
    comment="#",
    parse_dates=["Tanggal_Uji"]
)

print("--- Data Laboratorium Ter-Parsing Sempurna ---")
print(df_lab.info())
print("\nPreview Data:")
print(df_lab)


## 3. Ekstraksi dan Konsolidasi Lembar Kerja Ganda (*Multi-Sheet*) Excel
Dalam operasional agribisnis, laporan mingguan kerap disimpan dalam banyak lembar kerja (*sheet*) per divisi. Objek `pd.ExcelFile` memungkinkan pembacaan berkas sekali ke memori, kemudian lembar kerja dibaca secara berurutan tanpa proses buka-tutup berkas berulang kali.


In [ ]:
# Ingesti seluruh sheet dari workbook panen kebun
excel_obj = pd.ExcelFile(xlsx_path, engine="openpyxl")
print(f"Lembar kerja yang terdeteksi: {excel_obj.sheet_names}")

daftar_sheet = []
for sheet in excel_obj.sheet_names:
    df_temp = excel_obj.parse(sheet)
    df_temp["Nama_Afdeling"] = sheet
    daftar_sheet.append(df_temp)

df_panen_total = pd.concat(daftar_sheet, ignore_index=True)

print("\n--- Konsolidasi Seluruh Afdeling Sukses ---")
print(f"Total baris data terkonsolidasi: {len(df_panen_total)}")
print(df_panen_total.head(6))


## 4. Strategi Bongkahan (*Chunking Iterator*) untuk Dataset Skala Besar
Ketika dataset berukuran puluhan gigabita yang melebihi kapasitas RAM, pemanggilan `pd.read_csv()` tanpa parameter akan memicu *MemoryError* / OOM. Solusinya adalah memanfaatkan iterator bongkahan (`chunksize`).


In [ ]:
# Membuat dataset simulasi 50.000 transaksi timbangan
csv_besar_path = os.path.join(data_dir, "transaksi_skala_besar.csv")
n_rows = 50000
np.random.seed(42)

df_besar = pd.DataFrame({
    "ID_Truk": np.random.randint(100, 999, n_rows),
    "Kode_Blok": np.random.choice(["BK-01", "BK-02", "BK-03", "BK-04"], n_rows),
    "Berat_Netto_Kg": np.random.normal(6500, 800, n_rows).round(1)
})
df_besar.to_csv(csv_besar_path, index=False)
print(f"Dataset 50.000 baris tersimpan: {os.path.getsize(csv_besar_path) / 1024:.2f} KB")

# Agregasi bertahap menggunakan chunksize tanpa memuat seluruh file ke RAM
chunk_size = 10000
total_tonase = 0.0
total_truk_unik = set()
blok_tonase_map = {}

start_time = time.time()
for chunk in pd.read_csv(csv_besar_path, chunksize=chunk_size):
    total_tonase += chunk["Berat_Netto_Kg"].sum() / 1000.0
    total_truk_unik.update(chunk["ID_Truk"].unique())
    sub_grp = chunk.groupby("Kode_Blok")["Berat_Netto_Kg"].sum() / 1000.0
    for blok, ton in sub_grp.items():
        blok_tonase_map[blok] = blok_tonase_map.get(blok, 0.0) + ton

waktu_chunk = time.time() - start_time

print(f"Waktu Eksekusi Chunking: {waktu_chunk:.4f} detik")
print(f"Total Tonase Terhitung: {total_tonase:.2f} Ton")
print(f"Jumlah Armada Truk Terlibat: {len(total_truk_unik)}")
print("\nTotal Tonase per Blok:")
for k, v in sorted(blok_tonase_map.items()):
    print(f" - {k}: {v:.2f} Ton")


## 5. Benchmark Kinerja I/O: CSV vs Excel vs Apache Parquet
Format kolumnar biner *Apache Parquet* merupakan standar modern dalam *data pipeline* AI. Mari kita bandingkan ukuran penyimpanan (*storage footprint*), kecepatan penulisan (*write speed*), dan kecepatan pembacaan (*read speed*).


In [ ]:
# Siapkan data uji untuk benchmark
df_bench = pd.DataFrame({
    "Sensor_ID": np.random.randint(1, 100, 20000),
    "Suhu": np.random.uniform(20.0, 35.0, 20000),
    "Kelembaban": np.random.uniform(60.0, 95.0, 20000),
    "Tekanan": np.random.uniform(1000.0, 1025.0, 20000),
    "Status": np.random.choice(["Normal", "Warning", "Critical"], 20000)
})

hasil_bench = []

# 1. CSV
t0 = time.time()
csv_file = os.path.join(data_dir, "bench.csv")
df_bench.to_csv(csv_file, index=False)
t_w_csv = time.time() - t0

t0 = time.time()
_ = pd.read_csv(csv_file)
t_r_csv = time.time() - t0
size_csv = os.path.getsize(csv_file) / 1024

hasil_bench.append({"Format": "CSV", "Write_Time (s)": t_w_csv, "Read_Time (s)": t_r_csv, "Size (KB)": size_csv})

# 2. Excel (OpenPyXL)
t0 = time.time()
xlsx_file = os.path.join(data_dir, "bench.xlsx")
df_bench.to_excel(xlsx_file, index=False, engine="openpyxl")
t_w_xlsx = time.time() - t0

t0 = time.time()
_ = pd.read_excel(xlsx_file, engine="openpyxl")
t_r_xlsx = time.time() - t0
size_xlsx = os.path.getsize(xlsx_file) / 1024

hasil_bench.append({"Format": "Excel (XLSX)", "Write_Time (s)": t_w_xlsx, "Read_Time (s)": t_r_xlsx, "Size (KB)": size_xlsx})

# 3. Apache Parquet
t0 = time.time()
parquet_file = os.path.join(data_dir, "bench.parquet")
df_bench.to_parquet(parquet_file, index=False)
t_w_parq = time.time() - t0

t0 = time.time()
_ = pd.read_parquet(parquet_file)
t_r_parq = time.time() - t0
size_parq = os.path.getsize(parquet_file) / 1024

hasil_bench.append({"Format": "Apache Parquet", "Write_Time (s)": t_w_parq, "Read_Time (s)": t_r_parq, "Size (KB)": size_parq})

df_hasil = pd.DataFrame(hasil_bench)
print("=== HASIL BENCHMARK PERBANDINGAN FORMAT I/O ===")
print(df_hasil.to_string(index=False))


## 6. Tugas Tantangan Eksplorasi Mandiri (HOTS Challenge)

### Tantangan 1: Delimiter Sniffer & Auto-Correction
Implementasikan fungsi `baca_csv_cerdas(file_path)` yang secara dinamis:
1. Menentukan delimiter (`;`, `,`, `\t`) tanpa intervensi manual pengawas sistem.
2. Mengabaikan baris pendahuluan bertanda pagar.
3. Memilih separator desimal secara adaptif.


In [ ]:
import csv

def baca_csv_cerdas(file_path):
    with open(file_path, "r", encoding="latin-1") as f:
        sample_lines = []
        for _ in range(15):
            line = f.readline()
            if not line:
                break
            if not line.strip().startswith("#"):
                sample_lines.append(line)
        sample_text = "".join(sample_lines)
    
    sniffer = csv.Sniffer()
    dialect = sniffer.sniff(sample_text)
    sep_terdeteksi = dialect.delimiter
    
    df = pd.read_csv(
        file_path,
        sep=sep_terdeteksi,
        comment="#",
        encoding="latin-1",
        decimal="," if sep_terdeteksi == ";" else "."
    )
    return df, sep_terdeteksi

df_test, sep = baca_csv_cerdas(csv_nonstandar_path)
print(f"Auto-detection Berhasil! Pemisah terdeteksi: '{sep}'")
print(df_test.head(3))
